# Model Training using Custom Container in Vertex AI

## First, get a set of optimal hyperparameters.

In [1]:
from google.cloud import aiplatform
from datetime import datetime

### Hyperparameter Tuning Example

In [2]:
PROJECT_ID = "dw-analytics-d01"
REGION = "us"
ZONE = "central1"
LOCATION = f"{REGION}-{ZONE}"

bucket = 'ai-ml-vertex-d01'
table_name = 'dm_pc_tiny_data'
gcs_path = 'dm-propensity/pc'
segment = 106
timestamp = datetime.now().strftime('export_%Y%m%d_%H%M%S')

DISPLAY_NAME = (f"prop-model-{segment}-training")
TUNE_NAME = f"prop-{segment}-hypertune"
MODEL_TRAINING_IMAGE = "gcr.io/dw-analytics-d01/propimage:0.1-train"
BASE_OUTPUT_DIR = f"gs://{bucket}/{gcs_path}/{segment}/{timestamp}"
STAGING_BUCKET = (bucket)
MACHINE_TYPE = "n1-standard-4"

In [3]:
aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)

In [4]:
CMDARGS = [
    f"--segment={segment}",
    f"--table_name={table_name}",
    f"--timestamp={timestamp}",
    f"--hypertune",
]

machine_spec = {
    "machine_type": MACHINE_TYPE,
    "accelerator_type": None,
    "accelerator_count": 0,
}

worker_pool_spec = [
    {
        "replica_count": 1,
        "machine_spec": machine_spec,
        "container_spec": {
            "image_uri": MODEL_TRAINING_IMAGE,
            "args": CMDARGS,
        }
    }
]

In [5]:
c_job = aiplatform.CustomJob(
    display_name=DISPLAY_NAME,
    worker_pool_specs=worker_pool_spec,
    labels= {"run": "01"},
)

In [6]:
hpt = aiplatform.hyperparameter_tuning

hpt_job = aiplatform.HyperparameterTuningJob(
    display_name=TUNE_NAME,
    custom_job=c_job,
    metric_spec={"eval_auc": "maximize"},
    parameter_spec={
        "max_depth": hpt.IntegerParameterSpec(min=2, max=10, scale="linear"),
        "min_child_weight": hpt.IntegerParameterSpec(min=2, max=10, scale="linear"),
        "max_delta_step": hpt.DoubleParameterSpec(min=0.05, max=0.5, scale="linear"),
        "reg_lambda": hpt.DoubleParameterSpec(min=0.1, max=0.5, scale="linear"),
        "reg_alpha": hpt.DoubleParameterSpec(min=0.1, max=0.5, scale="linear"),
        "gamma": hpt.DoubleParameterSpec(min=0.001, max=0.05, scale="reverse_log"),
        "lr": hpt.DoubleParameterSpec(min=0.1, max=0.5, scale="log"),
    },
    max_trial_count=60,
    parallel_trial_count=4,
    max_failed_trial_count=4,
)

In [7]:
hpt_job.run(
    restart_job_on_worker_restart=True,
    service_account="dev-ana-ainb-admin@dw-analytics-d01.iam.gserviceaccount.com",
    # enable_web_access=True,
)

Creating HyperparameterTuningJob
HyperparameterTuningJob created. Resource name: projects/134453458552/locations/us-central1/hyperparameterTuningJobs/6391964943465316352
To use this HyperparameterTuningJob in another session:
hpt_job = aiplatform.HyperparameterTuningJob.get('projects/134453458552/locations/us-central1/hyperparameterTuningJobs/6391964943465316352')
View HyperparameterTuningJob:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/6391964943465316352?project=134453458552
HyperparameterTuningJob projects/134453458552/locations/us-central1/hyperparameterTuningJobs/6391964943465316352 current state:
JobState.JOB_STATE_PENDING
HyperparameterTuningJob projects/134453458552/locations/us-central1/hyperparameterTuningJobs/6391964943465316352 current state:
JobState.JOB_STATE_RUNNING
HyperparameterTuningJob projects/134453458552/locations/us-central1/hyperparameterTuningJobs/6391964943465316352 current state:
JobState.JOB_STATE_RUNNING
HyperparameterTuningJ

In [8]:
hpt_job.trials[31].parameters

[parameter_id: "gamma"
value {
  number_value: 0.042492185696248054
}
, parameter_id: "lr"
value {
  number_value: 0.37394962109224306
}
, parameter_id: "max_delta_step"
value {
  number_value: 0.5
}
, parameter_id: "max_depth"
value {
  number_value: 4.0
}
, parameter_id: "min_child_weight"
value {
  number_value: 4.0
}
, parameter_id: "reg_alpha"
value {
  number_value: 0.30993032603118775
}
, parameter_id: "reg_lambda"
value {
  number_value: 0.17127472432960333
}
]

## Then use best trial as input to model hyperparameters

In [7]:
# restart kernel
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

In [1]:
from google.cloud import aiplatform
from datetime import datetime

In [2]:
PROJECT_ID = "dw-analytics-d01"
REGION = "us"
ZONE = "central1"
LOCATION = f"{REGION}-{ZONE}"

bucket = 'ai-ml-vertex-d01'
table_name = 'dm_pc_tiny_data'
gcs_path = 'dm-propensity/pc'
segment = 106
timestamp = datetime.now().strftime('export_%Y%m%d_%H%M%S')

DISPLAY_NAME = (f"prop-model-{segment}-training")
MODEL_TRAINING_IMAGE = "gcr.io/dw-analytics-d01/propimage:0.1-train"
DEPLOY_IMAGE = "gcr.io/dw-analytics-d01/propimage:0.1-predict"
BASE_OUTPUT_DIR = f"gs://{bucket}/{gcs_path}/{segment}/{timestamp}"
STAGING_BUCKET = (bucket)

# DEPLOY_IMAGE = f"{REGION}-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.0-23:latest"

In [3]:
aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)

In [4]:
job = aiplatform.CustomContainerTrainingJob(
    display_name=DISPLAY_NAME,
    container_uri=MODEL_TRAINING_IMAGE,
    model_serving_container_image_uri=DEPLOY_IMAGE,
    model_serving_container_predict_route='/predict',
    model_serving_container_health_route='/health',
    labels={'version': 'v0_1'},
)

In [5]:
MACHINE_TYPE = "n1-standard-4"
MODEL_DISPLAY_NAME = (
    f"prop-{segment}-model"
)

max_depth = 3
min_child_weight = 4
reg_lambda = 0.1713
reg_alpha = 0.31
gamma = 0.0425
lr = 0.37395

CMDARGS = [
    f"--segment={segment}",
    f"--table_name={table_name}",
    f"--timestamp={timestamp}",
    f"--max_depth={max_depth}",
    f"--min_child_weight={min_child_weight}",
    f"--reg_lambda={reg_lambda}",
    f"--reg_alpha={reg_alpha}",
    f"--gamma={gamma}",
    f"--lr={lr}",
]

In [6]:
m_job = job.run(
    machine_type=MACHINE_TYPE,
    model_display_name=MODEL_DISPLAY_NAME, # don't provide a model display name if you don't have a prediction image
    model_labels={'version': 'v0_1'},
    args=CMDARGS,
    base_output_dir=BASE_OUTPUT_DIR,
    service_account="dev-ana-ainb-admin@dw-analytics-d01.iam.gserviceaccount.com",
    # enable_web_access=True,
    # sync=True,
)

Training Output directory:
gs://ai-ml-vertex-d01/dm-propensity/pc/106/export_20220713_232642 
View Training:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/2153646685551591424?project=134453458552
View backing custom job:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/9168750773088026624?project=134453458552
CustomContainerTrainingJob projects/134453458552/locations/us-central1/trainingPipelines/2153646685551591424 current state:
PipelineState.PIPELINE_STATE_RUNNING
CustomContainerTrainingJob projects/134453458552/locations/us-central1/trainingPipelines/2153646685551591424 current state:
PipelineState.PIPELINE_STATE_RUNNING
CustomContainerTrainingJob projects/134453458552/locations/us-central1/trainingPipelines/2153646685551591424 current state:
PipelineState.PIPELINE_STATE_RUNNING
CustomContainerTrainingJob projects/134453458552/locations/us-central1/trainingPipelines/2153646685551591424 current state:
PipelineState.PIPELINE_ST

In [7]:
# m_job.gca_resource

## Deploy to Endpoint only if Streaming predictions are required. Else, perform batch predictions.

### Performing Batch Predictions

In [8]:
storage_path = "gs://ai-ml-vertex-d01/dm-propensity/pc/106/data/sample.csv"
gcs_destination = "gs://ai-ml-vertex-d01/dm-propensity/pc/106"

In [9]:
date_time = datetime.now().strftime('predict_%Y%m%d_%H%M%S')
batch_display_name = f'model_{segment}_batch_test_{date_time}'

In [10]:
batch_prediction_job = m_job.batch_predict(
    job_display_name=batch_display_name,
    gcs_source=storage_path,
    instances_format="csv",
    gcs_destination_prefix=gcs_destination,
    machine_type=MACHINE_TYPE,
    starting_replica_count=1,
    max_replica_count=1,
    sync=True,
)

Creating BatchPredictionJob
BatchPredictionJob created. Resource name: projects/134453458552/locations/us-central1/batchPredictionJobs/7589491435787780096
To use this BatchPredictionJob in another session:
bpj = aiplatform.BatchPredictionJob('projects/134453458552/locations/us-central1/batchPredictionJobs/7589491435787780096')
View Batch Prediction Job:
https://console.cloud.google.com/ai/platform/locations/us-central1/batch-predictions/7589491435787780096?project=134453458552
BatchPredictionJob projects/134453458552/locations/us-central1/batchPredictionJobs/7589491435787780096 current state:
JobState.JOB_STATE_RUNNING
BatchPredictionJob projects/134453458552/locations/us-central1/batchPredictionJobs/7589491435787780096 current state:
JobState.JOB_STATE_RUNNING
BatchPredictionJob projects/134453458552/locations/us-central1/batchPredictionJobs/7589491435787780096 current state:
JobState.JOB_STATE_RUNNING
BatchPredictionJob projects/134453458552/locations/us-central1/batchPredictionJobs/

### Testing Model Locally